## Project: Concept Bottleneck Models on CelebA (Smiling)
In this project, you will study Concept Bottleneck Models (CBMs) and their variants on a real vision dataset. Using CelebA and Smiling as the prediction target, you will compare:

a standard image classifier (x → y)
a concept bottleneck model (x → c → y)
a hybrid CBM with a direct side channel (x → y) under varying side-channel dropout
You will analyze the tradeoff between accuracy, interpretability, and steerability.



Dataset
Use the CelebA dataset provided by torchvision.datasets.CelebA.

Inputs (x): RGB face images
Label (y): Smiling
Concepts (c): the following fixed subset of CelebA attributes (binary):
C_sub (10 concepts):

Mouth_Slightly_Open
High_Cheekbones
Chubby
Narrow_Eyes
Bags_Under_Eyes
Big_Lips
Big_Nose
Pointy_Nose
Bushy_Eyebrows
Arched_Eyebrows
Important: Do not include Smiling as a concept.

In [ ]:
%pip install pytorch-lightning;
%pip install torchmetrics;
%pip install torchvision;
%pip install torch;
%pip install matplotlib;
%pip install numpy;
%pip install pandas;
%pip install seaborn;

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
CELEBA_ROOT = Path("/content/drive/MyDrive")

- Use the official CelebA train/val/test split provided by torchvision
Resize images to a fixed resolution (e.g., 128×128)

- Normalize images using ImageNet statistics

Normalizing images before passing them into a pretrained model like ResNet is a preprocessing step that ensures your input data matches the distribution the model was originally trained on.
The process has two steps. First, scale raw pixel values from their original range of 0-255 down to 0-1 by dividing by 255. Second, normalize each channel using the ImageNet mean [0.485, 0.456, 0.406] and standard deviation [0.229, 0.224, 0.225], where each value corresponds to the R, G, and B channels respectively. The formula applied per channel is (x - mean) / std.
The reason you use ImageNet statistics specifically is that ResNet's weights were learned assuming inputs with that distribution. If you feed in data that looks statistically different, the activations throughout the network will be off, and the pretrained weights become less useful. In PyTorch this is handled cleanly with transforms.Normalize

In [ ]:
import os
import torch
from torchvision.datasets import CelebA
from torch.utils.data import Subset
from torchvision import transforms

# 1. Setup Transforms (from your document)
transform = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Define Root Path
CELEBA_ROOT = "/content/drive/MyDrive" 

# 3. Initialize the standard datasets (These contain the "Maps" of all 200k+ files)
# Note: We assign them to distinct variables for clarity
train_full = CelebA(root=CELEBA_ROOT, split="train", target_type="attr", transform=transform, download=False)
val_full = CelebA(root=CELEBA_ROOT, split="valid", target_type="attr", transform=transform, download=False)
test_full = CelebA(root=CELEBA_ROOT, split="test", target_type="attr", transform=transform, download=False)

# 4. Define the Filtering Function
def get_valid_celeba_indices(dataset, split_name):
    """
    Iterates through the dataset indices and checks if the corresponding 
    image file actually exists on the disk. Returns a list of valid indices.
    """
    # Construct the path to the image folder
    base_img_path = os.path.join(dataset.root, dataset.base_folder, "img_align_celeba")
    
    valid_indices = []
    total = len(dataset)
    print(f"Checking {total} files for {split_name} split...")

    for i in range(total):
        # Get the filename associated with index i
        filename = dataset.filename[i]
        path = os.path.join(base_img_path, filename)
        
        # Check existence
        if os.path.isfile(path):
            valid_indices.append(i)
            
    print(f"Found {len(valid_indices)} valid files out of {total} for {split_name}.")
    return valid_indices

# 5. Apply the Filter (This creates the "Safe" datasets)
# This part might take a minute or two to run because it pings Drive for every file.

print("--- Starting Drive File Validation ---")

# Filter Train
train_idx = get_valid_celeba_indices(train_full, "train")
dataset = Subset(train_full, train_idx) # This is your safe training set

# Filter Validation
val_idx = get_valid_celeba_indices(val_full, "valid")
val_dataset = Subset(val_full, val_idx) # This is your safe validation set

# Filter Test
test_idx = get_valid_celeba_indices(test_full, "test")
test_dataset = Subset(test_full, test_idx) # This is your safe test set

print("--- Validation Complete ---")
print(f"Final Train Size: {len(dataset)}")
print(f"Final Valid Size: {len(val_dataset)}")
print(f"Final Test Size:  {len(test_dataset)}")

In [ ]:
# from torchvision.datasets import CelebA
# from torchvision import transforms

# transform = transforms.Compose([
#     transforms.CenterCrop(178),
#     transforms.Resize((224, 224)), # Resnet with ImageNet trained on 224x224
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# dataset = CelebA(
#     root=str(CELEBA_ROOT),
#     split="all",
#     target_type="attr",
#     transform=transform,
#     download=False  # <--- we already have the files in Drive
# )

# val_dataset = CelebA(root=str(CELEBA_ROOT), split="valid", target_type="attr", transform=transform, download=False)
# test_dataset = CelebA(root=str(CELEBA_ROOT), split="test",  target_type="attr", transform=transform, download=False)



# print("Train size:", len(dataset))
# print("Valid size:", len(val_dataset))
# print("Test size:", len(test_dataset))

In [ ]:
# import os 
# from torch.utils.data import Subset

# def get_valid_celeba_indices(dataset):
#     """Indices where the image file exists."""
#     base = os.path.join(dataset.root, dataset.base_folder, "img_align_celeba")
#     valid = []
#     for i in range(len(dataset)):
#         path = os.path.join(base, dataset.filename[i])
#         if os.path.isfile(path):
#             valid.append(i)
#     return valid

# # After creating dataset, val_dataset, test_dataset:
# train_valid = get_valid_celeba_indices(dataset)
# dataset = Subset(dataset, train_valid)

# val_valid = get_valid_celeba_indices(val_dataset)
# val_dataset = Subset(val_dataset, val_valid)

# test_valid = get_valid_celeba_indices(test_dataset)
# test_dataset = Subset(test_dataset, test_valid)

In [ ]:
import matplotlib.pyplot as plt

def show_batch(dataset, num=8, figsize=(12, 6)):
    fig, axs = plt.subplots(2, 4, figsize=figsize)
    axs = axs.flatten()
    for i in range(min(num, len(dataset))):
        img, _ = dataset[i]
        if img.dim() == 3:
            img = img.permute(1, 2, 0)
        axs[i].imshow(img.clamp(0, 1))
        axs[i].axis("off")
    plt.tight_layout()
    plt.show()

show_batch(dataset, num=8)

In [ ]:
print(dataset.dataset.attr_names)

Report class balance for Smiling and each concept

In [ ]:
import seaborn as sns

LABEL_NAME = 'Smiling'
CONCEPT_NAMES = ["Mouth_Slightly_Open", "High_Cheekbones",
                 "Chubby", "Narrow_Eyes", "Bags_Under_Eyes",
                 "Big_Lips", "Big_Nose", "Pointy_Nose",
                 "Bushy_Eyebrows", "Arched_Eyebrows"]

names = [LABEL_NAME] + CONCEPT_NAMES
attr_names = list(dataset.dataset.attr_names)
idxs = [attr_names.index(n) for n in names]

props = dataset.dataset.attr[:, idxs].float().mean(dim=0)

for n, p in zip(names, props):
    print(f"{n:25s}  {p:.2%} positive")
    sns.barplot(x=names, y=props, color="steelblue")
plt.axhline(0.5, color="red", linestyle="--", alpha=0.5)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


Model architecture 
Use ResNet-18, available directly from torchvision.models.

Initialize with ImageNet pretrained weights
Remove the final classification layer
Use the shared backbone for all model variants
ResNet-18 is chosen because it is:

- lightweight (~11M parameters)
- well-understood and stable
- fast enough for repeated experiments (dropout sweeps)

#### Why reuse it rather than train from scratch?
ResNet-18 already knows how to detect edges, corners, textures, object parts. These low-level and mid-level features transfer well across vision tasks. Training that from scratch on your own dataset would require a lot more data and compute. You're essentially borrowing 1.2M images worth of learning for free.
#### Why remove the last layer?
The final fc layer is specific to ImageNet's 1000 classes. It's useless for your task. You want the 512-feature representation that precedes it, and then you'll attach your own task-specific head on top — whether that's a different number of output classes, a regression output, or in your case, variants with different dropout configurations.

In [ ]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
from torch import nn


resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

backbone = nn.Sequential(*list(resnet.children())[:-1]) 


x = torch.randn(8, 3, 224, 224) 
with torch.no_grad():
    out = backbone(x)

print(out.shape)

### 1) Baseline classifier (x → y)

**A standard image classifier: (x → y)**

- ResNet-18 backbone
- Single linear classification head

Report:

- Test Accuracy
- Test AUROC

In [ ]:
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score
import torch.nn.functional as F

SMILING_IDX = dataset.dataset.attr_names.index('Smiling')  # should be 31

class BaselineClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # strip fc
        self.classifier = nn.Linear(512, 1)  # single binary output

    def forward(self, x):
        features = self.backbone(x)
        features = features.flatten(start_dim=1)  # (B, 512)
        return self.classifier(features)           # (B, 1)


def evaluate(model, dataloader, device):
    model.eval()
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            smiling_labels = labels[:, SMILING_IDX].float()  # extract Smiling only

            logits = model(images).squeeze(1)  # (B,)
            probs = torch.sigmoid(logits)

            all_labels.append(smiling_labels.numpy())
            all_probs.append(probs.cpu().numpy())

    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)

    preds = (all_probs >= 0.5).astype(int)
    accuracy = (preds == all_labels).mean()
    auroc = roc_auc_score(all_labels, all_probs)

    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test AUROC:    {auroc:.4f}")

In [ ]:
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BaselineClassifier().to(device)

evaluate(model, DataLoader(test_dataset, batch_size=64, shuffle=False), device)